In [0]:
# Databricks notebook source
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, LongType

def profile_dataset(df: DataFrame, dataset_name: str):
    """
    Computes record count, prints schema, and generates null percentage summary per column.
    """
    print(f"\n{'='*30} {dataset_name} {'='*30}")
    total_rows = df.count()
    print(f"Total Rows: {total_rows:,} | Total Columns: {len(df.columns)}")
    
    # Inferred Schema
    print("\n--- Schema ---")
    df.printSchema()
    
    # Calculate null counts dynamically (escapes column names with backticks)
    null_exprs = [
        F.count(
            F.when(
                F.col(f"`{c}`").isNull() | F.isnan(f"`{c}`")
                if str(dtype) in ("double", "float")
                else F.col(f"`{c}`").isNull(),
                F.col(f"`{c}`")
            )
        ).alias(c)
        for c, dtype in df.dtypes
    ]
    
    null_row = df.agg(*null_exprs).collect()[0]
    
    # Format results for summary table
    summary_data = [
        (
            col_name,
            str(dtype),
            null_row[col_name],
            f"{(null_row[col_name] / total_rows * 100):.2f}%" if total_rows > 0 else "0.00%"
        )
        for col_name, dtype in df.dtypes
    ]
    
    summary_schema = StructType([
        StructField("Column Name", StringType(), False),
        StructField("Data Type", StringType(), False),
        StructField("Null Count", LongType(), False),
        StructField("Null Percentage", StringType(), False)
    ])
    
    summary_df = spark.createDataFrame(summary_data, schema=summary_schema)
    display(summary_df)

VOLUME_PATH = "/Volumes/vstone/landing/raw_data/chunks"

# 1. Chunk 1 - Initial CSV
df_chunk1 = (spark.read.format("csv")
             .option("header", "true")
             .option("inferSchema", "true")
             .load(f"{VOLUME_PATH}/chunk1_initial.csv.gz"))
profile_dataset(df_chunk1, "Chunk 1: Initial CSV (.gz)")

# 2. Chunk 2 - Incremental CSV
df_chunk2 = (spark.read.format("csv")
             .option("header", "true")
             .option("inferSchema", "true")
             .load(f"{VOLUME_PATH}/chunk2_incremental.csv.gz"))
profile_dataset(df_chunk2, "Chunk 2: Incremental CSV (.gz)")

# 3. Chunk 3 - JSON (Lines)
df_chunk3 = (spark.read.format("json")
             .load(f"{VOLUME_PATH}/chunk3.json.gz"))
profile_dataset(df_chunk3, "Chunk 3: JSON (.gz)")

# 4. Chunk 4 - XML
df_chunk4 = (spark.read.format("xml")
             .option("rowTag", "record")
             .load(f"{VOLUME_PATH}/chunk4.xml.gz"))
profile_dataset(df_chunk4, "Chunk 4: XML (.gz)")